## UKB BNF Lookup brand names exploration

In [ ]:
import pandas as pd
import re
# import json

#### Helpers

In [ ]:
def tokenize_drug_name(name: str):
    name = ' ' + name + ' '
    name = re.sub(r'(?<=\d),(?=\d{3})', '', name)
    name = re.sub(r'(?<!\d)\.(?=\d)', '0.', name)
    name = re.sub(r'(?<=\d)\s+%', '%', name)
    name = re.sub(r'((?<!\d)\.|\.(?!\d))', ' ', name)
    name = re.sub(r'(?<!\d)%', ' ', name)
    name = re.sub(r'[\s!"#&\'()*+,\-/:;<=>?@\[\\\]^_`|~]+', ' ', name)
    if name == ' ':
        return None
    return name.lower()

#### BNF Lookup loading

In [ ]:
df = pd.read_csv('../../../data/ukb_lkps/bnf_lkp.csv', dtype=str)
df = df[['BNF_Presentation', 'BNF_Product']]

In [ ]:
display(df)

#### Empty rows

In [ ]:
display(df[df.isna().any(axis=1)])
print(len(df[df.isna().any(axis=1)]))

In [ ]:
df = df.dropna()

In [ ]:
len(df)

#### Distinct entries

In [ ]:
df['BNF_Presentation'] = df['BNF_Presentation'].str.strip()
df['BNF_Product'] = df['BNF_Product'].str.strip()

Unique rows:

In [ ]:
uniq_rows = df.drop_duplicates()
print(len(uniq_rows))
df = uniq_rows

Unique BNF Products:

In [ ]:
len(df['BNF_Product'].apply(tokenize_drug_name).unique())

#### Underscore testing

In [ ]:
display(df[(df['BNF_Presentation'].str.count("_") > 1)])
wo_underscore = df[(df['BNF_Presentation'].str.count("_") < 1)]
print(len(wo_underscore))
wo_underscore = wo_underscore[~wo_underscore["BNF_Product"].str.contains("DUMMY PRODUCT")]
display(wo_underscore)

In [ ]:
df = df[(df['BNF_Presentation'].str.count("_") == 1)]
len(df)

#### Filtering special/dummy records

In [ ]:
filtered_df = df[~df["BNF_Product"].str.startswith("DUMMY PRODUCT ")]
filtered_df = filtered_df[~filtered_df["BNF_Product"].str.startswith('Proprietary Co Prepn Bnf')]
len(filtered_df)

### Parentheses in `BNF_Product`

In [ ]:
len(df[(df['BNF_Product'].str.count("\(") > 1) | (df['BNF_Product'].str.count("\)") > 1)])

In [ ]:
def get_parentheses_content(text):
    match = re.search(r"\((.+)\)", text)
    if not match:
        return None
    return match.group(1)

parentheses_content = df['BNF_Product'].apply(get_parentheses_content).dropna()
print(len(parentheses_content))
print(len(parentheses_content.unique()))
print(len(parentheses_content.apply(tokenize_drug_name).unique()))

In [ ]:
content_freq = parentheses_content.value_counts().reset_index()
content_freq.columns = ["content", "count"]
display(content_freq.head(30))
display(content_freq.tail(5))
content_freq.to_csv('bnf_product_parentheses_suffix_freq.csv', index=False)

#### Informative (non-naming) suffixes

In [ ]:
suf = [tokenize_drug_name(i) for i in list(dict.fromkeys([
        'Systemic', 'Parent', 'Eye', 'Proprietary Preps', 'Gel', 'Inj', 'Soln', 'Buccal', 'Nsl', 'Top', 'Oral', 'Mth', 'Rectal', 'Scalp',
        'Cap', 'Inj', 'Flushes', 'Crm', 'Inj', 'Sach', 'Syr', 'Tab', 'Cap', 'Inj', 'Tab', 'Gel', 'Pharmacia', 'Suppos', 'Tab', 'Oint', 'Spy', 'Cap',
        'Inj', 'Susp', 'Tab', 'Liq', 'Loz', 'Spy', 'Inj', 'Tab', 'Inf', 'Syr', 'Pi', 'Pastil', 'Anal', 'Oint', 'Ear', 'Vag', 'Blad', 'Inh'
    ]))]
len(suf)

In [ ]:
print(suf)

### Underscores in `BNF_Presentation`

#### Content after underscore (mostly not-naming, informative)

In [ ]:
def get_underscore_content(text):
    match = re.search(r"_(.+?) ", text)
    if not match:
        return None
    return match.group(1)

underscore_content = filtered_df['BNF_Presentation'].apply(get_underscore_content).dropna()
print(len(underscore_content))
print(len(underscore_content.unique()))
print(len(underscore_content.apply(tokenize_drug_name).unique()))

In [ ]:
content_freq = underscore_content.value_counts().reset_index()
content_freq.columns = ["content", "count"]
display(content_freq.head(200))
display(content_freq.tail(5))

#### Does content before underscore similar to `BNF_Product` (brand name)?

In [ ]:
def get_before_underscore(text):
    match = re.search(r"^(.+?)_", text)
    if not match:
        return None
    return match.group(1)


filtered_df['before_underscore'] = filtered_df['BNF_Presentation'].apply(get_before_underscore).apply(tokenize_drug_name)
filtered_df['tokenized_BNF_Product'] = filtered_df['BNF_Product'].apply(tokenize_drug_name)
filtered_df

In [ ]:
substring_matched = filtered_df[~(filtered_df['before_underscore'].isna()) 
                                & (filtered_df['before_underscore'].str.len() > 2)
                                & (filtered_df.apply(lambda x: x["before_underscore"] in x["tokenized_BNF_Product"], axis=1))]
print(f'{len(substring_matched)} of {len(filtered_df)} ({len(substring_matched) * 100.0 / len(filtered_df):.1f}%)')
substring_matched

In [ ]:
substring_matched = filtered_df[~(filtered_df['before_underscore'].isna()) 
                                & (filtered_df['before_underscore'].str.len() > 2)
                                & (filtered_df['before_underscore'] != filtered_df["tokenized_BNF_Product"])
                                & (filtered_df.apply(lambda x: x["before_underscore"] in x["tokenized_BNF_Product"], axis=1))]
print(f'{len(substring_matched)} of {len(filtered_df)} ({len(substring_matched) * 100.0 / len(filtered_df):.1f}%)')
substring_matched

In [ ]:
substring_matched = filtered_df[(filtered_df['before_underscore'].isna()) 
                                | ~(filtered_df.apply(lambda x: x["before_underscore"] in x["tokenized_BNF_Product"], axis=1))]
print(f'{len(substring_matched)} of {len(filtered_df)} ({len(substring_matched) * 100.0 / len(filtered_df):.1f}%)')
substring_matched